# BQuant Export v2 — Earnings Dates & Ex-Dividends

Produces two CSVs for the IBKR overnight study:

| File | Columns | Method |
|---|---|---|
| `earnings_dates.csv` | `ticker, ann_date` | `is_eps(fpt='Q', fpo=-8..0)` → `revision_date` |
| `exdiv.csv` | `ticker, ex_date, amount` | total return − price return |

## How these were found

Direct field names did not work. `dvd_hist_all`, `dvd_hist`, `eqy_dvd_hist_gross` do not exist;
`dvd_ex_dt` returns one row per ticker with no amount; `dividends` with a date range broadcasts a
single **PROJECTED** future event across every calendar day. `announcement_dt` returns only the
latest date.

Two indirect routes work and are used here.

**Earnings** — `is_eps` with fiscal-period parameters returns one row per quarter, and its
`revision_date` is the date that quarter's EPS was first reported: the announcement date. Note BQL
uses `fpt` / `fpo`, not `fa_period_type` / `fa_period_offset`.

**Dividends** — `day_to_day_tot_return_gross_dvds` minus the price return isolates the dividend on
exactly the ex-date. Validated on IBM: 7 ex-dates over 20 months, implied amounts 1.67–1.69 against
an actual quarterly dividend of 1.67–1.69, dates matching the known Feb/May/Aug/Nov schedule.

## Why this matters

IB `TRADES` bars are split-adjusted but **not dividend-adjusted**, so every ex-date is a fake
overnight gap. And 102 of the overnight moves above 10% in the IBKR panel are earnings reactions
sitting in the decile extremes — precisely where a long/short book takes its positions.

## Running it

Run all cells; expect several minutes for 500 names. Download both CSVs from the file browser,
copy to `~/ib_work/`, then:

    python ib_loader_v2.py build --exdiv exdiv.csv

In [2]:
# ============================ Cell 1 — Setup
import pandas as pd
import numpy as np
import bql

bq = bql.Service()
members = bq.univ.members("SPX Index")

START, END = "2024-08-01", "2026-08-22"
CHUNK = 50          # tickers per BQL request


def tidy(item, label=None):
    d = item.df().reset_index()
    d.columns = [str(c).strip().lower() for c in d.columns]
    d = d.rename(columns={"id": "ticker"})
    d["ticker"] = d["ticker"].astype(str).str.split().str[0]
    return d


def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


# bq.univ.members() returns a BqlItem (a server-side query object), not a list.
# Resolve it to concrete tickers so the requests below can be chunked.
_resp = bq.execute(bql.Request(members, {"nm": bq.data.name()}))
_ids = list(_resp)[0].df().reset_index()
_ids.columns = [str(c).strip().lower() for c in _ids.columns]
_ids = _ids.rename(columns={"id": "ticker"})

FULL = sorted(_ids["ticker"].astype(str).unique().tolist())   # "AAPL US Equity"
ALL = [t.split()[0] for t in FULL]                            # "AAPL"

print(f"window   : {START} -> {END}")
print(f"{len(FULL)} tickers resolved")
print(FULL[:5])

window   : 2024-08-01 -> 2026-08-22
503 tickers resolved
['A UN Equity', 'AAPL UW Equity', 'ABBV UN Equity', 'ABNB UW Equity', 'ABT UN Equity']


In [3]:
# ============================ Cell 2 — Earnings announcement dates
# is_eps(fpt='Q', fpo=range(-8,0)) returns one row per quarter; revision_date is
# the date that quarter's EPS was first reported.
print("=" * 68)
print("EARNINGS ANNOUNCEMENT DATES")
print("=" * 68)

frames = []
for i, grp in enumerate(chunks(FULL, CHUNK), 1):
    try:
        resp = bq.execute(bql.Request(grp, {
            "eps": bq.data.is_eps(fpt="Q", fpo=bq.func.range(-10, 0))}))
        d = tidy(list(resp)[0], "eps")
        frames.append(d[["ticker", "revision_date"]])
    except Exception as e:
        print(f"  chunk {i}: {type(e).__name__}: {str(e)[:90]}")
    print(f"  earnings chunk {i}/{(len(FULL)-1)//CHUNK+1}", end="\r")

earn = pd.concat(frames, ignore_index=True)
earn["ann_date"] = pd.to_datetime(earn["revision_date"], errors="coerce").dt.normalize()
earn = earn[["ticker", "ann_date"]].dropna().drop_duplicates()
earn = earn[(earn["ann_date"] >= START) & (earn["ann_date"] <= END)]
earn = earn.sort_values(["ticker", "ann_date"]).reset_index(drop=True)

per = earn.groupby("ticker").size()
print(f"\nrows           : {len(earn):,}")
print(f"tickers        : {earn['ticker'].nunique():,}")
print(f"dates/ticker   : median {per.median():.0f}  (expect ~8 for two years)")
print(f"range          : {earn['ann_date'].min().date()} -> {earn['ann_date'].max().date()}")
if per.median() < 6:
    print("WARNING: under ~6 per ticker means incomplete history; the filter will under-flag.")
earn.to_csv("earnings_dates.csv", index=False)
print("wrote earnings_dates.csv")
display(earn.head())

EARNINGS ANNOUNCEMENT DATES
  earnings chunk 11/11
rows           : 3,853
tickers        : 501
dates/ticker   : median 8  (expect ~8 for two years)
range          : 2024-08-01 -> 2026-08-21
wrote earnings_dates.csv


,ticker,ann_date
0,A,2025-02-26
1,A,2025-06-02
2,A,2025-08-29
3,A,2025-12-19
4,A,2026-02-25


In [4]:
# ============================ Cell 3 — Ex-dividends from total return
# total return - price return = dividend yield, on exactly the ex-date.
# Validated on IBM: 7 events, implied 1.67-1.69 vs actual 1.67-1.69.
print("=" * 68)
print("EX-DIVIDEND DATES AND AMOUNTS")
print("=" * 68)

RNG = bq.func.range(START, END)
MIN_EXCESS = 0.0005        # 5bp floor separates dividends from rounding noise

rows = []
for i, grp in enumerate(chunks(FULL, CHUNK), 1):
    try:
        resp = bq.execute(bql.Request(grp, {
            "tr": bq.data.day_to_day_tot_return_gross_dvds(dates=RNG),
            "px": bq.data.px_last(dates=RNG, fill="NA")}))
        items = list(resp)
        tr = tidy(items[0], "tr")[["ticker", "date", "tr"]]
        px = tidy(items[1], "px")[["ticker", "date", "px"]]
        d = tr.merge(px, on=["ticker", "date"])
        d["date"] = pd.to_datetime(d["date"])
        d["tr"] = pd.to_numeric(d["tr"], errors="coerce")
        d["px"] = pd.to_numeric(d["px"], errors="coerce")
        d = d.dropna(subset=["px"]).sort_values(["ticker", "date"])

        g = d.groupby("ticker")
        d["px_ret"] = g["px"].pct_change()
        d["prev_px"] = g["px"].shift(1)
        d["excess"] = d["tr"] - d["px_ret"]
        hit = d[(d["excess"] > MIN_EXCESS) & d["prev_px"].notna()].copy()
        hit["amount"] = (hit["excess"] * hit["prev_px"]).round(4)
        rows.append(hit[["ticker", "date", "amount"]].rename(columns={"date": "ex_date"}))
    except Exception as e:
        print(f"  chunk {i}: {type(e).__name__}: {str(e)[:90]}")
    print(f"  dividend chunk {i}/{(len(FULL)-1)//CHUNK+1}", end="\r")

dvd = pd.concat(rows, ignore_index=True).drop_duplicates()
dvd = dvd[(dvd["amount"] > 0.005) & (dvd["amount"] < 50)]
dvd = dvd.sort_values(["ticker", "ex_date"]).reset_index(drop=True)

per = dvd.groupby("ticker").size()
print(f"\nrows            : {len(dvd):,}")
print(f"tickers         : {dvd['ticker'].nunique():,}")
print(f"events/ticker   : median {per.median():.0f}  (expect ~8 quarterly, ~24 monthly)")
print(f"amount range    : {dvd['amount'].min():.3f} -> {dvd['amount'].max():.3f}")
print(f"median amount   : {dvd['amount'].median():.3f}")
dvd.to_csv("exdiv.csv", index=False)
print("wrote exdiv.csv")
display(dvd.head(10))

EX-DIVIDEND DATES AND AMOUNTS
  dividend chunk 11/11
rows            : 3,223
tickers         : 402
events/ticker   : median 8  (expect ~8 quarterly, ~24 monthly)
amount range    : 0.010 -> 13.600
median amount   : 0.600
wrote exdiv.csv


,ticker,ex_date,amount
0,A,2024-10-01,0.236
1,A,2024-12-31,0.248
2,A,2025-04-01,0.248
3,A,2025-07-01,0.248
4,A,2025-09-30,0.248
5,A,2026-01-06,0.255
6,A,2026-03-31,0.255
7,A,2026-06-30,0.255
8,AAPL,2024-08-12,0.250
9,AAPL,2024-11-08,0.250


In [5]:
# ============================ Cell 4 — Sanity checks
print("=" * 68)
print("VALIDATION")
print("=" * 68)

# IBM is the reference case: known 1.67-1.69 quarterly, Feb/May/Aug/Nov.
ibm = dvd[dvd["ticker"] == "IBM"]
print("IBM ex-dividends (expect ~1.67-1.69 quarterly):")
print(ibm.to_string(index=False) if len(ibm) else "   IBM not in universe")

print("\nAmount distribution — implausible values indicate a bad extraction:")
print(dvd["amount"].describe().round(3).to_string())

big = dvd[dvd["amount"] > 5]
print(f"\namounts over 5.00: {len(big)} "
      f"(special dividends are real; check a few)")
if len(big):
    print(big.head(8).to_string(index=False))

print("\nEarnings dates per ticker:")
print(earn.groupby("ticker").size().describe().round(1).to_string())

print("\nBoth files written. Download from the file browser, copy to ~/ib_work/, then:")
print("    python ib_loader_v2.py build --exdiv exdiv.csv")

VALIDATION
IBM ex-dividends (expect ~1.67-1.69 quarterly):
ticker    ex_date  amount
   IBM 2024-08-09    1.67
   IBM 2024-11-12    1.67
   IBM 2025-02-10    1.67
   IBM 2025-05-09    1.68
   IBM 2025-08-08    1.68
   IBM 2025-11-10    1.68
   IBM 2026-02-10    1.68
   IBM 2026-05-08    1.69
   IBM 2026-08-10    1.69

Amount distribution — implausible values indicate a bad extraction:
count    3223.000
mean        0.768
std         0.692
min         0.010
25%         0.320
50%         0.600
75%         1.000
max        13.600

amounts over 5.00: 14 (special dividends are real; check a few)
ticker    ex_date  amount
   BLK 2024-09-09    5.10
   BLK 2024-12-05    5.10
   BLK 2025-03-07    5.21
   BLK 2025-06-05    5.21
   BLK 2025-09-05    5.21
   BLK 2025-12-05    5.21
   BLK 2026-03-06    5.73
   BLK 2026-06-05    5.73

Earnings dates per ticker:
count    501.0
mean       7.7
std        1.7
min        1.0
25%        8.0
50%        8.0
75%        9.0
max       10.0

Both files written. 

In [6]:
from IPython.display import FileLink, display
import os

for fn in ("earnings_dates.csv", "exdiv.csv"):
    if os.path.exists(fn):
        print(f"{fn}  ({os.path.getsize(fn)/1024:.1f} KB)")
        display(FileLink(fn))
    else:
        print(f"{fn}  NOT FOUND")

earnings_dates.csv  (57.2 KB)


/project/earnings_dates.csv

exdiv.csv  (63.6 KB)


/project/exdiv.csv

In [7]:
import base64, os
from IPython.display import HTML, display

def dl(fn):
    if not os.path.exists(fn):
        return f"<p>{fn} NOT FOUND</p>"
    b64 = base64.b64encode(open(fn, "rb").read()).decode()
    kb = os.path.getsize(fn) / 1024
    return (f'<p><a download="{fn}" href="data:text/csv;base64,{b64}" '
            f'style="font-size:15px;padding:8px 14px;background:#1f6feb;color:#fff;'
            f'border-radius:6px;text-decoration:none;display:inline-block;margin:4px 0">'
            f'⬇ Download {fn} ({kb:.1f} KB)</a></p>')

display(HTML(dl("earnings_dates.csv") + dl("exdiv.csv")))